# HW-KNN: Predicting a Customer's Card Plan

A credit card company offers three card plans: **Basic**, **Plus** and **Premium**. The data has 30 existing customers: their annual income, their credit score, and the plan each one holds. You will use KNN to predict the plan of a new customer from `AnnualIncome` and `CreditScore`.

**The homework has 10 multiple-choice problems.** Problems 1 to 3 are conceptual problems: no code is required. Problems 4 to 10 require brief coding: each one comes with starter code from the lecture, and you fill in the missing pieces.

**Where to get this file.** This notebook, the homework instruction file, can be downloaded from the course GitHub repository, under **KNN**.

**Where to answer.** These 10 problems are also the 10 questions of the **HW-KNN quiz on Canvas**, in the same order and with the same answer choices. Use this notebook to work out your answers, then enter them in the Canvas quiz; only the Canvas quiz is graded. Each question is worth **10 points**, for **100 points** in total. See Canvas for the due date.

**You must scale the inputs before you use KNN, just like in the lecture.** KNN uses distance, and income (tens of thousands) has much bigger numbers than credit score (hundreds). Use `StandardScaler`:

- call `fit_transform()` **once**, on the 30 customers
- call `transform()` (not `fit_transform()`) on every new customer
- fit the model on the **scaled** data, and scale every new customer before you call `predict()`, `kneighbors()` or `predict_proba()`

Use `k = 4` (`n_neighbors=4`) unless a problem gives a different `k`.

## How to use this notebook

- Run the cells **in order**, from top to bottom (Shift+Enter runs a cell). Later cells use the variables created in earlier cells.
- **Problems 1 to 3:** read the problem and choose an answer. No code is needed.
- **Problems 4 to 10:** each one comes with starter code. Replace every `____` with your own code, then run the cell. A cell that still contains `____` stops with an error.
- Each problem ends with a "Your answer" cell. Double-click it and type A, B, C or D to keep track of your answer. When you are done, enter your answers in the HW-KNN quiz on Canvas.
- If you change a cell, run the cells below it again. If your results look strange, restart the notebook and run all the cells again from the top.
- The code follows the lecture notebook step by step, so keep the lecture open while you work.

## Setup

Run this cell first. It is the same as in the lecture.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# Print numpy arrays with 3 decimals, to keep the outputs readable.
np.set_printoptions(precision=3, suppress=True)

## The data

The next three cells create the data and show it. They are complete: just run them.

In [ ]:
# Create one simple customer dataset.
# The points are intentionally not perfectly separated.

customer_df = pd.DataFrame({
    "AnnualIncome": [
        36000, 48000, 89000, 55000, 73000, 66000,
        30000, 99000, 125000, 60000, 40000, 58000,
        70000, 86000, 110000, 47000, 72000, 74000,
        83000, 61000, 92000, 58000, 93000, 118000,
        42000, 52000, 81000, 76000, 78000, 103000
    ],
    "CreditScore": [
        610, 700, 790, 640, 715, 780,
        645, 695, 765, 680, 720, 800,
        595, 705, 750, 620, 650, 805,
        605, 705, 730, 600, 680, 790,
        655, 730, 770, 620, 745, 780
    ],
    "CardPlan": [
        "Basic", "Plus", "Premium", "Basic", "Plus", "Premium",
        "Basic", "Plus", "Premium", "Basic", "Plus", "Premium",
        "Basic", "Plus", "Premium", "Basic", "Plus", "Premium",
        "Basic", "Plus", "Premium", "Basic", "Plus", "Premium",
        "Basic", "Plus", "Premium", "Basic", "Plus", "Premium"
    ]
})

customer_df

This is a small **hand-made dataset** created only for practicing KNN.

The last column, `CardPlan`, is the **label**: the plan we already know for each customer.

In [ ]:
# Check how many customers hold each plan.
customer_df["CardPlan"].value_counts()

In [ ]:
# One color for each plan. We reuse these colors in every plot.
colors = {
    "Basic": "tab:blue",
    "Plus": "tab:orange",
    "Premium": "tab:green"
}

plt.figure(figsize=(7, 5))

# Plot each group of customers separately.
for plan, color in colors.items():
    subset = customer_df[customer_df["CardPlan"] == plan]
    plt.scatter(subset["AnnualIncome"], subset["CreditScore"],
                c=color, edgecolors="black", label=plan)

plt.xlabel("Annual Income")
plt.ylabel("Credit Score")
plt.title("The 30 customers: the plan of every customer is known")
plt.legend()
plt.show()

Each dot is one customer, and the color shows the plan.

Look at the two axes. Income runs from 30,000 to 125,000, while credit score runs from about 600 to 800. Income has much bigger numbers, which is why we must scale the inputs.

## Part 1: Concepts (no code needed)

### Problem 1

Which statement explains why KNN is a **supervised** learning method?

- **A.** It uses the distance between customers.
- **B.** Every customer in the data already has a known label (`CardPlan`), and `fit()` receives both `X` and `y`.
- **C.** The inputs must be scaled before the model is fitted.
- **D.** We must choose `k` before the model is fitted.

**Your answer to Problem 1:** 

### Problem 2

With `k = 4`, the 4 nearest neighbors of a new customer hold these plans, closest first: Premium, Plus, Plus, Plus. Which statement is correct?

- **A.** KNN predicts Premium, because the closest neighbor holds Premium.
- **B.** KNN cannot make a prediction, because the 4 neighbors do not all hold the same plan.
- **C.** KNN predicts Plus, because 3 of the 4 neighbors hold Plus. With `k = 1` it would predict Premium.
- **D.** KNN predicts Plus with `k = 4`, and it would also predict Plus with `k = 1`.

**Your answer to Problem 2:** 

### Problem 3

Suppose we do **not** scale the inputs, so KNN applies the distance formula to the raw numbers.

- Customer A: income 60,000, credit score 700
- Customer B: income 60,000, credit score 600
- Customer C: income 61,000, credit score 700

**Hint.** KNN decides which customers are "nearest" with the ordinary (Euclidean) distance formula (Section 5 of the lecture). For two customers with incomes $x_1$ and $x_2$ and credit scores $y_1$ and $y_2$:

$$\text{distance} = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$$

In words: take the difference in income and the difference in credit score, square each difference, add the two squares, and take the square root of the sum. The formula only adds up squared differences; it does not know what units the numbers are in.

For A and B, $x_1 = 60{,}000$, $y_1 = 700$, $x_2 = 60{,}000$ and $y_2 = 600$:

$$\text{distance}(A, B) = \sqrt{(60{,}000 - 60{,}000)^2 + (700 - 600)^2} = \sqrt{0^2 + 100^2} = \sqrt{0 + 10{,}000} = \sqrt{10{,}000} = 100$$

The order of the subtraction does not matter, because each difference is squared.

Now compute distance(A, C) in the same way, and compare it with distance(A, B).

Which statement is correct?

- **A.** B is closer to A (distance 100) than C is (distance 1,000). Income has the bigger numbers, so it controls the distance and the credit score hardly counts.
- **B.** B is closer to A (distance 100) than C is (distance 1,000). This shows that the credit score controls the distance.
- **C.** C is closer to A than B is, because C has the same credit score as A.
- **D.** B and C are equally close to A, because each differs from A in only one input.

**Your answer to Problem 3:** 

## Part 2: Coding

Problems 4 to 10 use the 30 customers. `X` is `AnnualIncome` and `CreditScore`, and `y` is `CardPlan`.

Problems 5, 6, 7, 8 and 10 are about one **new customer** who applies for a card, with `AnnualIncome = 50,000` and `CreditScore = 676`. You create this customer in Problem 5 and use the same customer again in Problems 6, 7, 8 and 10. Problem 9 is about two other customers, given in that problem.

Work through the problems in order. Each coding cell uses the variables created in the cells above it.

First we define `X` and `y`. This cell is complete: just run it.

In [ ]:
# X = the INPUTS: what we know about each customer.
X = customer_df[["AnnualIncome", "CreditScore"]]

# y = the ANSWERS: the plan each customer holds (the label).
y = customer_df["CardPlan"]

print("X shape:", X.shape)
print("y shape:", y.shape)

### Problem 4

Scale `X` with `StandardScaler`. What do `scaler.mean_` and `scaler.scale_` show?

- **A.** Mean of each feature: `[0. 0.]`, Std of each feature: `[1. 1.]`
- **B.** Mean of each feature: `[23949.6 65.546]`, Std of each feature: `[72500. 702.5]`
- **C.** Mean of each feature: `[70111.111 46638.889]`, Std of each feature: `[14908.569 16376.054]`
- **D.** Mean of each feature: `[72500. 702.5]`, Std of each feature: `[23949.6 65.546]`

In [ ]:
# Lecture: Section 6, "Scaling: fit_transform() vs. transform()"
scaler = StandardScaler()

# Learn the mean and std of each column from the 30 customers, and scale them.
X_scaled = scaler.____(X)

# The scaler remembers what it learned:
print("Mean of each feature:", scaler.mean_)
print("Std of each feature: ", scaler.scale_)

**Your answer to Problem 4:** 

### Problem 5

A **new customer** applies for a card: `AnnualIncome = 50,000` and `CreditScore = 676`. Create this new customer as a DataFrame and scale it. What are the scaled values?

- **A.** `[[50000 676]]`
- **B.** `[[-0.939 -0.404]]`
- **C.** `[[0. 0.]]`
- **D.** `[[0.939 0.404]]`

In [ ]:
# Lecture: Section 6, "Scaling: fit_transform() vs. transform()" and "What does predict() return?"
# The new customer: AnnualIncome = 50,000 and CreditScore = 676.
# Create it as a DataFrame with the same two column names as X.
new_customer = pd.DataFrame({
    "AnnualIncome": [____],
    "CreditScore": [____]
})

# Scale it with the SAME mean and std as the 30 customers.
new_customer_scaled = scaler.____(new_customer)
print("Scaled new customer:", new_customer_scaled)

# Tip: if you correct a mistake in this cell, run the Problem 4 cell again first.

**Your answer to Problem 5:** 

### Problem 6

Fit KNN with `k = 4` on the scaled data. Which plan does it predict for the new customer from Problem 5 (income 50,000, credit score 676)?

- **A.** Basic
- **B.** Plus
- **C.** Premium
- **D.** It is a tie between Basic and Plus, 2 votes each.

In [ ]:
# Lecture: Section 6, "What does fit() do?" and "What does predict() return?"
# Create the KNN model with k = 4.
knn = KNeighborsClassifier(n_neighbors=____)

# fit(): hand the model the scaled inputs AND the answers.
knn.fit(____, ____)

# predict() returns one label per input row, as an array.
predicted_class = knn.predict(____)
print("predict() returned:", predicted_class)
print("Predicted plan:", predicted_class[0])    # [0] takes the first element

**Your answer to Problem 6:** 

### Problem 7

Use `knn.kneighbors()` to find the 4 nearest neighbors of the new customer from Problem 5 (income 50,000, credit score 676). Which row positions does `indices` contain, closest first?

- **A.** 8, 23, 14, 29
- **B.** 1, 25, 15, 3
- **C.** 1, 9, 24, 3
- **D.** 4, 19, 13, 9

In [ ]:
# Lecture: Section 6, "Where did the prediction come from? kneighbors()"
# Ask the model for the 4 nearest neighbors of the new customer.
distances, indices = knn.kneighbors(____)

print("distances:", distances)
print("indices:  ", indices)

# Look up those rows in the original table, and show the distances next to them.
neighbors = customer_df.iloc[indices[0]].copy()
neighbors["ScaledDistance"] = distances[0].round(3)
neighbors

**Your answer to Problem 7:** 

**Optional picture (not graded).** After you finish Problem 7, run the next cell. It shows the new customer (black star) and its 4 nearest neighbors (circled). The cell is complete: just run it.

In [ ]:
plt.figure(figsize=(7, 5))

for plan, color in colors.items():
    subset = customer_df[customer_df["CardPlan"] == plan]
    plt.scatter(subset["AnnualIncome"], subset["CreditScore"],
                c=color, edgecolors="black", label=plan)

# The new customer.
plt.scatter(new_customer["AnnualIncome"], new_customer["CreditScore"],
            marker="*", s=350, c="black", label="New customer")

# Circle its 4 nearest neighbors.
plt.scatter(neighbors["AnnualIncome"], neighbors["CreditScore"],
            s=300, facecolors="none", edgecolors="black", linewidths=2,
            label="4 nearest neighbors")

plt.xlabel("Annual Income")
plt.ylabel("Credit Score")
plt.title("KNN prediction for the new customer (k = 4)")
plt.legend()
plt.show()

### Problem 8

Use `knn.predict_proba()` for the new customer from Problem 5 (income 50,000, credit score 676). It returns the fraction of the 4 neighbors that hold each plan, in the order Basic, Plus, Premium. What are the three fractions?

- **A.** Basic 0.25, Plus 0.75, Premium 0
- **B.** Basic 0.667, Plus 0.333, Premium 0
- **C.** Basic 0, Plus 0, Premium 1
- **D.** Basic 0.75, Plus 0.25, Premium 0

In [ ]:
# Lecture: Section 6, the cell after kneighbors() ("The vote")
# The order of the plans in the output:
print("Order of the plans:", knn.classes_)

# The fraction of the 4 neighbors that hold each plan.
print("Fractions:         ", knn.predict_proba(____))

**Your answer to Problem 8:** 

### Problem 9

Predict two more customers with **one** call to `predict()`:

- Customer 1: income 62,000, credit score 710
- Customer 2: income 110,000, credit score 780

Which plans does `predict()` return?

- **A.** Customer 1: Premium, Customer 2: Premium
- **B.** Customer 1: Plus, Customer 2: Premium
- **C.** Customer 1: Premium, Customer 2: Plus
- **D.** Customer 1: Basic, Customer 2: Premium

In [ ]:
# Lecture: Section 6, "Predicting several customers at once"
two_customers = pd.DataFrame({
    "AnnualIncome": [____, ____],
    "CreditScore":  [____, ____]
})

# Same recipe: scale with the SAME scaler, then predict. One label per row.
two_customers_scaled = scaler.____(two_customers)
print(knn.predict(____))

**Your answer to Problem 9:** 

### Problem 10

So far we used `k = 4`. Does the choice of `k` matter? Predict the new customer from Problem 5 (income 50,000, credit score 676) again with `k` = 1, 3, 5, 7, 9, 11 and 15, as in Section 7 of the lecture. For which of these values does KNN predict **Plus**?

- **A.** None of them. KNN predicts Basic for every `k`.
- **B.** `k` = 1 and 3
- **C.** `k` = 1, 7 and 11
- **D.** All seven. KNN predicts Plus for every `k`.

In [ ]:
# Lecture: Section 7, "Does the choice of k matter?"
print("New customer (income 50,000, credit score 676):")
for k in [1, 3, 5, 7, 9, 11, 15]:
    knn_k = KNeighborsClassifier(n_neighbors=____)
    knn_k.fit(____, ____)
    print(f"  k = {k:2d}: {knn_k.predict(____)[0]}")

**Your answer to Problem 10:** 

## Before you finish

- Check that every coding cell runs without an error and that every "Your answer" cell has one letter (A, B, C or D).
- Enter your 10 answers in the **HW-KNN quiz on Canvas**. Question 1 on Canvas is Problem 1 here, Question 2 is Problem 2, and so on.
- The choices on Canvas are the same as here and in the same order, but Canvas does not show the letters A to D. Pick the choice whose text matches your answer.